In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, round, desc, row_number, dense_rank
from pyspark.sql.window import Window
import requests
import json

spark = SparkSession.builder \
    .appName("Lab2_ClickHouse_Reports") \
    .config("spark.driver.memory", "2g") \
    .config("spark.jars", "/home/jovyan/jars/postgresql-42.7.1.jar") \
    .getOrCreate()

pg_url = "jdbc:postgresql://postgres:5432/bigdata_lab"
pg_props = {"user": "student", "password": "student123", "driver": "org.postgresql.Driver"}

df_date = spark.read.jdbc(url=pg_url, table="dim_date", properties=pg_props)
df_customer = spark.read.jdbc(url=pg_url, table="dim_customer", properties=pg_props)
df_seller = spark.read.jdbc(url=pg_url, table="dim_seller", properties=pg_props)
df_product = spark.read.jdbc(url=pg_url, table="dim_product", properties=pg_props)
df_store = spark.read.jdbc(url=pg_url, table="dim_store", properties=pg_props)
df_supplier = spark.read.jdbc(url=pg_url, table="dim_supplier", properties=pg_props)
df_fact = spark.read.jdbc(url=pg_url, table="fact_sale", properties=pg_props)

print("Все таблицы загружены из PostgreSQL")

def write_to_clickhouse(df, table_name):
    rows = df.toJSON().collect()
    data = "\n".join(rows)
    
    columns = ", ".join([f"{f.name} {_map_type(f.dataType)}" for f in df.schema.fields])
    
    create_sql = f"DROP TABLE IF EXISTS reports.{table_name}; CREATE TABLE reports.{table_name} ({columns}) ENGINE = MergeTree() ORDER BY tuple()"
    requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                  params={"database": "reports", "query": create_sql})
    
    requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                  params={"database": "reports", "query": f"INSERT INTO reports.{table_name} FORMAT JSONEachRow"},
                  data=data.encode("utf-8"))
    print(f"{table_name}: {len(rows)} строк записано в ClickHouse")

def _map_type(spark_type):
    type_str = str(spark_type).lower()
    if "int" in type_str: return "Int64"
    if "double" in type_str or "float" in type_str: return "Float64"
    if "decimal" in type_str: return "Decimal(38,2)"
    if "date" in type_str: return "Date"
    if "timestamp" in type_str: return "DateTime"
    return "String"

print("Функции готовы")

Все таблицы загружены из PostgreSQL
Функции готовы


In [2]:
df_prod_report = df_fact.join(df_product, "product_key", "left")

top10_products = df_prod_report.groupBy("product_id", "product_name", "product_category") \
    .agg(
        sum("sale_quantity").alias("total_quantity"),
        sum("sale_total_price").alias("total_revenue"),
        avg("product_rating").alias("avg_rating"),
        avg("product_reviews").alias("avg_reviews")
    ) \
    .orderBy(desc("total_revenue")).limit(10)

print("Топ-10 продуктов по выручке:")
top10_products.select("product_name", "total_revenue", "total_quantity").show(truncate=False)

revenue_by_category = df_prod_report.groupBy("product_category") \
    .agg(sum("sale_total_price").alias("total_revenue")) \
    .orderBy(desc("total_revenue"))

product_mart = df_prod_report.groupBy("product_id", "product_name", "product_category") \
    .agg(
        sum("sale_quantity").alias("total_quantity"),
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        round(avg("product_rating"), 2).alias("avg_rating"),
        avg("product_reviews").cast("int").alias("total_reviews")
    ) \
    .withColumnRenamed("product_id", "product_id") \
    .withColumnRenamed("product_name", "product_name") \
    .withColumnRenamed("product_category", "product_category")

write_to_clickhouse(product_mart, "product_sales_mart")

Топ-10 продуктов по выручке:
+------------+------------------+--------------+
|product_name|total_revenue     |total_quantity|
+------------+------------------+--------------+
|Bird Cage   |4005.9800000000005|63            |
|Cat Toy     |3784.44           |59            |
|Bird Cage   |3751.09           |60            |
|Bird Cage   |3682.52           |65            |
|Bird Cage   |3645.94           |59            |
|Dog Food    |3616.93           |44            |
|Dog Food    |3571.1            |63            |
|Cat Toy     |3548.8599999999997|48            |
|Cat Toy     |3546.42           |44            |
|Bird Cage   |3537.0400000000004|52            |
+------------+------------------+--------------+

product_sales_mart: 1000 строк записано в ClickHouse


In [3]:
df_cust_report = df_fact.join(df_customer, "customer_key", "left")

customer_mart = df_cust_report.groupBy("customer_id", "first_name", "last_name", "customer_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_spent"),
        count("sale_id").alias("purchase_count"),
        round(avg("sale_total_price"), 2).alias("avg_check")
    ) \
    .withColumnRenamed("customer_country", "country")

print("Топ-10 клиентов:")
customer_mart.orderBy(desc("total_spent")).select("first_name", "last_name", "total_spent", "country").show(10, truncate=False)

write_to_clickhouse(customer_mart, "customer_sales_mart")

Топ-10 клиентов:
+----------+----------+-----------+-------------+
|first_name|last_name |total_spent|country      |
+----------+----------+-----------+-------------+
|Hannie    |Braddon   |4005.98    |China        |
|Mercy     |Antonomoli|3784.44    |Laos         |
|Genni     |Schultze  |3751.09    |United States|
|Herschel  |Chaff     |3682.52    |United States|
|Ramsay    |Karran    |3645.94    |Philippines  |
|Ariadne   |Silverman |3616.93    |Thailand     |
|Beckie    |Dunkerton |3571.1     |Pakistan     |
|Consuelo  |Poge      |3548.86    |China        |
|Paige     |Pacher    |3546.42    |Portugal     |
|Bink      |Shelford  |3537.04    |Thailand     |
+----------+----------+-----------+-------------+
only showing top 10 rows

customer_sales_mart: 1000 строк записано в ClickHouse


In [4]:
df_time_report = df_fact.join(df_date, "date_key", "left")

time_mart = df_time_report.groupBy("year", "month", "month_name", "quarter") \
    .agg(
        round(sum("sale_total_price"), 2).alias("monthly_revenue"),
        count("sale_id").alias("order_count"),
        round(avg("sale_total_price"), 2).alias("avg_order_size")
    ) \
    .orderBy("year", "month")

print("Месячные тренды:")
time_mart.select("year", "month_name", "monthly_revenue", "order_count", "avg_order_size").show(10, truncate=False)

write_to_clickhouse(time_mart, "time_sales_mart")

Месячные тренды:
+----+----------+---------------+-----------+--------------+
|year|month_name|monthly_revenue|order_count|avg_order_size|
+----+----------+---------------+-----------+--------------+
|2021|January   |224158.54      |874        |256.47        |
|2021|February  |192348.31      |739        |260.28        |
|2021|March     |207282.2       |843        |245.89        |
|2021|April     |206592.82      |837        |246.83        |
|2021|May       |211764.86      |828        |255.75        |
|2021|June      |215042.8       |822        |261.61        |
|2021|July      |220496.51      |858        |256.99        |
|2021|August    |221275.78      |897        |246.68        |
|2021|September |210623.43      |839        |251.04        |
|2021|October   |228743.32      |892        |256.44        |
+----+----------+---------------+-----------+--------------+
only showing top 10 rows

time_sales_mart: 12 строк записано в ClickHouse


In [5]:
df_store_report = df_fact.join(df_store, "store_key", "left")

store_mart = df_store_report.groupBy("store_name", "store_city", "store_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        count("sale_id").alias("order_count"),
        round(avg("sale_total_price"), 2).alias("avg_check")
    )

print("Топ-5 магазинов:")
store_mart.orderBy(desc("total_revenue")).select("store_name", "store_city", "total_revenue").show(5, truncate=False)

write_to_clickhouse(store_mart, "store_sales_mart")

Топ-5 магазинов:
+----------+----------+-------------+
|store_name|store_city|total_revenue|
+----------+----------+-------------+
|Mynte     |Brunflo   |15751.71     |
|Quatz     |Gemuruh   |15176.64     |
|Jayo      |Colima    |13976.01     |
|Quinu     |Bélabo    |13952.88     |
|Realcube  |Cincinnati|13700.77     |
+----------+----------+-------------+
only showing top 5 rows

store_sales_mart: 383 строк записано в ClickHouse


In [7]:
# Поставщики
df_supp_report = df_fact.join(df_product.select("product_key", "supplier_key"), "product_key", "left") \
    .join(df_supplier, "supplier_key", "left")

supplier_mart = df_supp_report.groupBy("supplier_name", "supplier_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        count("sale_id").alias("sales_count")
    )

print("Топ-5 поставщиков:")
supplier_mart.orderBy(desc("total_revenue")).show(5, truncate=False)

write_to_clickhouse(supplier_mart, "supplier_sales_mart")

Топ-5 поставщиков:
+-------------+----------------+-------------+-----------+
|supplier_name|supplier_country|total_revenue|sales_count|
+-------------+----------------+-------------+-----------+
|NULL         |NULL            |2412919.15   |9550       |
|Flashdog     |Russia          |19089.69     |70         |
|Ozu          |Philippines     |13865.79     |50         |
|Divanoodle   |China           |13028.36     |50         |
|Brainbox     |Israel          |11528.62     |40         |
+-------------+----------------+-------------+-----------+
only showing top 5 rows

supplier_sales_mart: 17 строк записано в ClickHouse


In [8]:
# Качество продукции
df_quality = df_fact.join(df_product, "product_key", "left")

quality_mart = df_quality.groupBy("product_id", "product_name", "product_category", "product_rating", "product_reviews") \
    .agg(
        sum("sale_quantity").alias("total_sold"),
        round(sum("sale_total_price"), 2).alias("total_revenue")
    )

print("Топ-5 по рейтингу:")
quality_mart.orderBy(desc("product_rating")).select("product_name", "product_rating", "product_reviews", "total_sold").show(5, truncate=False)

print("Худшие 5 по рейтингу:")
quality_mart.orderBy("product_rating").select("product_name", "product_rating", "product_reviews", "total_sold").show(5, truncate=False)

write_to_clickhouse(quality_mart, "product_quality_mart")

Топ-5 по рейтингу:
+------------+--------------+---------------+----------+
|product_name|product_rating|product_reviews|total_sold|
+------------+--------------+---------------+----------+
|Cat Toy     |5.0           |882            |51        |
|Dog Food    |5.0           |757            |60        |
|Dog Food    |5.0           |821            |53        |
|Cat Toy     |5.0           |177            |44        |
|Bird Cage   |5.0           |678            |60        |
+------------+--------------+---------------+----------+
only showing top 5 rows

Худшие 5 по рейтингу:
+------------+--------------+---------------+----------+
|product_name|product_rating|product_reviews|total_sold|
+------------+--------------+---------------+----------+
|Dog Food    |1.0           |133            |45        |
|Cat Toy     |1.0           |591            |47        |
|Dog Food    |1.0           |726            |65        |
|Dog Food    |1.0           |319            |74        |
|Cat Toy     |1.0     

In [12]:
def write_to_clickhouse_v3(df, table_name):
    rows = df.toJSON().collect()
    data = "\n".join(rows)
    
    columns = ", ".join([f"{f.name} {_map_type(f.dataType)}" for f in df.schema.fields])
    
    # DROP отдельно
    resp = requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                        data=f"DROP TABLE IF EXISTS reports.{table_name}")
    
    # CREATE отдельно
    resp = requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                        data=f"CREATE TABLE reports.{table_name} ({columns}) ENGINE = MergeTree() ORDER BY tuple()")
    if resp.status_code != 200:
        print(f"Ошибка CREATE {table_name}: {resp.text.strip()}")
        return
    
    # INSERT
    resp = requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                        params={"query": f"INSERT INTO reports.{table_name} FORMAT JSONEachRow"},
                        data=data.encode("utf-8"))
    if resp.status_code != 200:
        print(f"Ошибка INSERT {table_name}: {resp.text.strip()}")
    else:
        print(f"{table_name}: {len(rows)} строк OK")

# Перезаписываем все витрины
product_mart = df_fact.join(df_product, "product_key", "left") \
    .groupBy("product_id", "product_name", "product_category") \
    .agg(
        sum("sale_quantity").alias("total_quantity"),
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        round(avg("product_rating"), 2).alias("avg_rating"),
        avg("product_reviews").cast("int").alias("total_reviews")
    )
write_to_clickhouse_v3(product_mart, "product_sales_mart")

customer_mart = df_fact.join(df_customer, "customer_key", "left") \
    .groupBy("customer_id", "first_name", "last_name", "customer_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_spent"),
        count("sale_id").alias("purchase_count"),
        round(avg("sale_total_price"), 2).alias("avg_check")
    ) \
    .withColumnRenamed("customer_country", "country")
write_to_clickhouse_v3(customer_mart, "customer_sales_mart")

time_mart = df_fact.join(df_date, "date_key", "left") \
    .groupBy("year", "month", "month_name", "quarter") \
    .agg(
        round(sum("sale_total_price"), 2).alias("monthly_revenue"),
        count("sale_id").alias("order_count"),
        round(avg("sale_total_price"), 2).alias("avg_order_size")
    )
write_to_clickhouse_v3(time_mart, "time_sales_mart")

store_mart = df_fact.join(df_store, "store_key", "left") \
    .groupBy("store_name", "store_city", "store_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        count("sale_id").alias("order_count"),
        round(avg("sale_total_price"), 2).alias("avg_check")
    )
write_to_clickhouse_v3(store_mart, "store_sales_mart")

df_supp_report = df_fact.join(df_product.select("product_key", "supplier_key"), "product_key", "left") \
    .join(df_supplier, "supplier_key", "left")
supplier_mart = df_supp_report.groupBy("supplier_name", "supplier_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        count("sale_id").alias("sales_count")
    )
write_to_clickhouse_v3(supplier_mart, "supplier_sales_mart")

quality_mart = df_fact.join(df_product, "product_key", "left") \
    .groupBy("product_id", "product_name", "product_category", "product_rating", "product_reviews") \
    .agg(
        sum("sale_quantity").alias("total_sold"),
        round(sum("sale_total_price"), 2).alias("total_revenue")
    )
write_to_clickhouse_v3(quality_mart, "product_quality_mart")

print("\nГотово")

product_sales_mart: 1000 строк OK
customer_sales_mart: 1000 строк OK
time_sales_mart: 12 строк OK
store_sales_mart: 383 строк OK
supplier_sales_mart: 17 строк OK
product_quality_mart: 1000 строк OK

Готово


In [13]:
import requests

# Проверяем таблицы в БД reports
resp = requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                    params={"database": "reports", "query": "SHOW TABLES"})
print("Таблицы в reports:")
print(resp.text)

# Проверяем количество строк в каждой витрине
tables_ch = ["product_sales_mart", "customer_sales_mart", "time_sales_mart", 
             "store_sales_mart", "supplier_sales_mart", "product_quality_mart"]

for t in tables_ch:
    resp = requests.post("http://clickhouse:8123/", auth=("student", "student123"),
                        params={"database": "reports", "query": f"SELECT count() FROM {t}"})
    if resp.status_code == 200:
        print(f"{t}: {resp.text.strip()} строк")
    else:
        print(f"{t}: ОШИБКА - {resp.text.strip()[:100]}")

Таблицы в reports:
customer_sales_mart
product_quality_mart
product_sales_mart
store_sales_mart
supplier_sales_mart
time_sales_mart

product_sales_mart: 1000 строк
customer_sales_mart: 1000 строк
time_sales_mart: 12 строк
store_sales_mart: 383 строк
supplier_sales_mart: 17 строк
product_quality_mart: 1000 строк
